# A/B Testing Pipeline — Inspection Model Comparison

This notebook demonstrates the **A/B testing module**: loading two models,
running them on the same images, and comparing their predictions.

Use case: before deploying a new model version to the inspection line,
compare it against the current production model to measure:
- **Disagreement rate** — % of images where they give different answers
- **Confidence delta** — how different are they in certainty?
- **Per-class agreement** — are they consistent on specific defect types?

---

**Notebook sections:**
1. Generate synthetic images + two model variants
2. Run A/B comparison
3. Metrics: disagreement rate, confidence delta
4. Per-class agreement breakdown
5. Deep-dive: which images cause the most disagreement?

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style='whitegrid')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cpu


## 1. Synthetic data + two model variants

We reuse the same synthetic image generator from notebook 01,
and create **two ResNet18 models** with different random seeds — 
simulating Model A (production) vs Model B (new challenger).

In [2]:
import tempfile, pathlib
from PIL import Image

TMP_DIR = pathlib.Path(tempfile.mkdtemp()) / 'steeldefect_ab_demo'
TMP_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 64
N_IMAGES = 200
CLASS_NAMES = ['scratch', 'pit', 'roll_mark', 'edge_crack', 'clean']
N_CLASSES = len(CLASS_NAMES)

def make_defect(rng, cls_id):
    img = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32) * 0.72
    if cls_id == 0:   # scratch
        for r in rng.choice(IMG_SIZE, size=rng.integers(2, 6), replace=False):
            img[r, :] = rng.uniform(0.1, 0.4)
    elif cls_id == 1:  # pit
        cx, cy = rng.integers(10, IMG_SIZE-10, size=2)
        r = rng.integers(4, 12)
        Y, X = np.ogrid[:IMG_SIZE, :IMG_SIZE]
        img[(X-cx)**2 + (Y-cy)**2 <= r**2] = rng.uniform(0.1, 0.3)
    elif cls_id == 2:  # roll mark — diagonal stripe
        for d in range(-8, IMG_SIZE+8):
            for i in range(IMG_SIZE):
                j = d + i
                if 0 <= j < IMG_SIZE and rng.random() < 0.5:
                    img[i, j] = rng.uniform(0.1, 0.35)
    elif cls_id == 3:  # edge crack
        img[:, :rng.integers(5, 15)] = rng.uniform(0.0, 0.2)
    # cls_id == 4: clean, just noise
    img += rng.normal(0, 0.03, img.shape)
    return np.clip(img, 0, 1)

rng = np.random.default_rng(123)
saved_paths = []

for i in range(N_IMAGES):
    cls_id = i % N_CLASSES
    arr = (make_defect(rng, cls_id) * 255).astype(np.uint8)
    fname = TMP_DIR / f'img_{i:04d}_cls{cls_id}.png'
    Image.fromarray(arr).save(fname)
    saved_paths.append(str(fname))

print(f'Generated {len(saved_paths)} images in {TMP_DIR}')

Generated 200 images in C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\steeldefect_ab_demo


In [3]:
from steeldefect_active_learning.models.model_loader import load_model

# Model A — production (seed 1)
torch.manual_seed(1)
model_a = load_model(arch='resnet18', checkpoint=None, num_classes=N_CLASSES, device=DEVICE)

# Model B — challenger (seed 99, different random init)
torch.manual_seed(99)
model_b = load_model(arch='densenet121', checkpoint=None, num_classes=N_CLASSES, device=DEVICE)

print('Model A (ResNet18) and Model B (DenseNet121) loaded.')

[model_loader] No checkpoint provided for 'resnet18' — using random weights.
[model_loader] No checkpoint provided for 'densenet121' — using random weights.
Model A (ResNet18) and Model B (DenseNet121) loaded.


## 2. Run A/B Comparison

In [4]:
from steeldefect_active_learning.data.dataset import SteelDefectDataset
from torch.utils.data import DataLoader
from steeldefect_active_learning.ab_testing.ab_pipeline import run_ab_comparison

dataset = SteelDefectDataset(source=str(TMP_DIR), image_size=(64, 64))
loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

ab_df = run_ab_comparison(
    model_a=model_a,
    model_b=model_b,
    dataloader=loader,
    task='classification',
    device=DEVICE,
)

print(f'Results for {len(ab_df)} images')
ab_df.head(8)

Results for 200 images


,image_path,pred_a,confidence_a,pred_b,confidence_b,agrees,confidence_delta
0,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.335605,0,0.223669,False,0.111936
1,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.327926,1,0.219242,False,0.108684
2,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,2,0.382809,0,0.241069,False,0.141740
3,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.340861,0,0.223620,False,0.117241
4,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.328598,4,0.218911,True,0.109687
5,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.355604,0,0.228863,False,0.126741
6,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.343251,1,0.217566,False,0.125686
7,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,2,0.371908,0,0.244144,False,0.127765


## 3. Global Metrics

In [5]:
from steeldefect_active_learning.ab_testing.metrics import (
    disagreement_rate, confidence_delta_summary
)

dr = disagreement_rate(ab_df)
delta = confidence_delta_summary(ab_df)

print(f'Disagreement rate : {dr:.1%}')
print(f'Confidence delta  : mean={delta["mean"]:.3f}  '
      f'median={delta["median"]:.3f}  '
      f'p90={delta["p90"]:.3f}  '
      f'max={delta["max"]:.3f}')

Disagreement rate : 73.5%
Confidence delta  : mean=0.120  median=0.116  p90=0.143  max=0.163


In [6]:
# Agreement bar chart
agree_n = ab_df['agrees'].sum()
disagree_n = len(ab_df) - agree_n

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: agree/disagree counts
axes[0].bar(['Agree', 'Disagree'], [agree_n, disagree_n],
            color=['#4CAF50', '#F44336'], edgecolor='white', width=0.5)
axes[0].set_title(f'Overall Agreement\n(disagreement = {dr:.1%})')
axes[0].set_ylabel('Number of images')
for bar, val in zip(axes[0].patches, [agree_n, disagree_n]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontweight='bold')

# Right: confidence delta histogram
axes[1].hist(ab_df['confidence_delta'], bins=30, color='#2196F3', edgecolor='white')
axes[1].axvline(delta['mean'], color='red', linestyle='--',
                label=f'mean = {delta["mean"]:.3f}')
axes[1].axvline(delta['p90'], color='orange', linestyle=':',
                label=f'p90 = {delta["p90"]:.3f}')
axes[1].set_xlabel('|confidence_A − confidence_B|')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence Delta Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

C:\Users\amrou\AppData\Local\Temp\ipykernel_21888\3383440144.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Per-class Agreement Breakdown

For which defect classes do the models agree most / least?

In [7]:
from steeldefect_active_learning.ab_testing.metrics import per_class_agreement

class_stats = per_class_agreement(ab_df)
class_stats['class_name'] = class_stats['class_id'].map(
    {i: n for i, n in enumerate(CLASS_NAMES)}
)
print(class_stats)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#4CAF50' if r >= 0.8 else '#FF9800' if r >= 0.5 else '#F44336'
          for r in class_stats['agreement_rate']]
bars = ax.bar(class_stats['class_name'], class_stats['agreement_rate'],
              color=colors, edgecolor='white')
ax.axhline(0.8, color='green', linestyle='--', alpha=0.6, label='80% threshold')
ax.set_ylim(0, 1.1)
ax.set_xlabel('Defect class')
ax.set_ylabel('Agreement rate')
ax.set_title('Per-class Agreement: Model A vs Model B')
ax.legend()
for bar, (_, row) in zip(bars, class_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{row["agreement_rate"]:.0%}\n(n={row["count"]})',
            ha='center', fontsize=8)
plt.tight_layout()
plt.show()

   class_id  agreement_rate  count class_name
0         2        0.000000     27  roll_mark
1         4        0.306358    173      clean


C:\Users\amrou\AppData\Local\Temp\ipykernel_21888\2558168097.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Deep-dive: Most Disagreed Images

Inspect the images where A and B give the most different predictions.  
These are the best candidates for **manual review / arbitration**.

In [8]:
# Top 10 most-disputed images (highest confidence delta AND disagrees=True)
disputed = ab_df[ab_df['agrees'] == False].sort_values('confidence_delta', ascending=False)
top_disputed = disputed.head(10)

print(f'Disputed images: {len(disputed)} / {len(ab_df)}')
top_disputed[['image_path', 'pred_a', 'confidence_a', 'pred_b', 'confidence_b', 'confidence_delta']]

Disputed images: 147 / 200


,image_path,pred_a,confidence_a,pred_b,confidence_b,confidence_delta
167,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.418500,0,0.255274,0.163226
62,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,2,0.408408,0,0.246782,0.161627
147,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.406099,0,0.244878,0.161221
93,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.384418,0,0.224337,0.160080
177,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.404220,0,0.246112,0.158108
8,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.382284,0,0.224788,0.157496
187,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.398037,0,0.241754,0.156283
172,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.408061,0,0.252284,0.155777
102,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,4,0.400179,0,0.245457,0.154722
127,C:\Users\amrou\AppData\Local\Temp\tmpnvu6tzcf\...,2,0.406225,0,0.252192,0.154033


In [9]:
# Visualise the top 6 most disputed images
n_show = min(6, len(top_disputed))
fig, axes = plt.subplots(1, n_show, figsize=(n_show * 2.5, 3))
if n_show == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, top_disputed.head(n_show).iterrows()):
    img = Image.open(row['image_path'])
    ax.imshow(img)
    ax.set_title(
        f"A={CLASS_NAMES[row['pred_a']]}\nB={CLASS_NAMES[row['pred_b']]}\n"
        f"Δconf={row['confidence_delta']:.2f}",
        fontsize=8
    )
    ax.axis('off')

plt.suptitle('Most Disputed Images (A ≠ B)', fontsize=12)
plt.tight_layout()
plt.show()

C:\Users\amrou\AppData\Local\Temp\ipykernel_21888\2909113212.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Confidence scatter: Model A vs Model B

Points on the diagonal = both models equally confident. Points far from it = divergence.

In [10]:
fig, ax = plt.subplots(figsize=(6, 6))

agrees_mask = ab_df['agrees'].values.astype(bool)
ax.scatter(ab_df.loc[agrees_mask, 'confidence_a'],
           ab_df.loc[agrees_mask, 'confidence_b'],
           alpha=0.5, s=20, color='#4CAF50', label='Agree')
ax.scatter(ab_df.loc[~agrees_mask, 'confidence_a'],
           ab_df.loc[~agrees_mask, 'confidence_b'],
           alpha=0.7, s=30, color='#F44336', label='Disagree', marker='x')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect agreement')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('Confidence — Model A')
ax.set_ylabel('Confidence — Model B')
ax.set_title('Confidence Scatter: A vs B')
ax.legend()
plt.tight_layout()
plt.show()

C:\Users\amrou\AppData\Local\Temp\ipykernel_21888\2422677737.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Save full report + plots

In [11]:
from steeldefect_active_learning.ab_testing.metrics import plot_comparison

os.makedirs('../outputs', exist_ok=True)
ab_df.to_csv('../outputs/ab_report_demo.csv', index=False)

plot_comparison(ab_df, output_dir='../outputs/plots', task='classification', prefix='demo_ab')

print('Saved: outputs/ab_report_demo.csv')
print('Saved: outputs/plots/demo_ab_*.png')

Saved: outputs/ab_report_demo.csv
Saved: outputs/plots/demo_ab_*.png


## Summary

| Metric | Value |
|--------|-------|
| Total images | 200 |
| Disagreement rate | see above |
| Mean confidence delta | see above |
| Welch t-test | Section 8 below |
| Cohen's d effect size | Section 8 below |

**Decision rule (example):**
- If disagreement rate > 30% → investigate before deploying Model B.
- If disagreement rate < 10% AND mean confidence delta < 0.05 → safe to deploy.
- If Cohen's |d| < 0.2 → negligible difference, safe to deploy with monitoring.
- Always review the top-10 most disputed images manually.

**Statistical framework (Quin & Weyns 2023 — AutoPABS):**
- Welch's t-test (unequal variance) for confidence distributions
- Cohen's d for effect size: < 0.2 negligible, 0.2–0.5 small, 0.5–0.8 medium, ≥ 0.8 large
- Power analysis: required n for 80% power at observed effect size

## Welch's t-test + Cohen's d Effect Size (Quin & Weyns 2023 — AutoPABS)

**Hypothesis test** (two-tailed):
- $H_0$: $\mu_A = \mu_B$ (no difference between models)
- $H_1$: $\mu_A \neq \mu_B$

**Welch's t-test** (unequal variance): more robust than Student's t-test when $\sigma^2_A \neq \sigma^2_B$.

**Cohen's d** interpretation:
| |d| range | Magnitude |
|----------|----------|
| < 0.2    | negligible|
| 0.2–0.5  | small     |
| 0.5–0.8  | medium    |
| ≥ 0.8    | large     |

**Power analysis**: $n_{required} = 2(z_{\alpha/2} + z_\beta)^2 \sigma^2 / \Delta^2$

In [ ]:
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from steeldefect_active_learning.ab_testing.metrics import (
    statistical_significance_test, power_analysis, required_sample_size
)

np.random.seed(42)
# Simulate two models with a small but real difference
conf_a = np.random.beta(8, 2, 500)   # Model A: better calibrated
conf_b = np.random.beta(6, 2, 500)   # Model B: slightly less confident
df_test = pd.DataFrame({
    "confidence_a":   conf_a,
    "confidence_b":   conf_b,
    "agrees":         (np.round(conf_a) == np.round(conf_b)).astype(int),
    "confidence_delta": np.abs(conf_a - conf_b),
})

stat = statistical_significance_test(df_test, alpha=0.05)
pwr  = power_analysis(df_test, alpha=0.05)
req_n = required_sample_size(effect_size=0.05, sigma=0.15)

print("=== Welch's t-test Results (Quin & Weyns 2023) ===")
print(f"  t-statistic    : {stat['t_stat']:.4f}")
print(f"  p-value        : {stat['p_value']:.6f}")
print(f"  Significant    : {stat['significant']}")
print(f"  Cohen's d      : {stat['cohens_d']:.4f}  ({stat['effect_magnitude']})")
print(f"  95% CI (A-B)   : ({stat['confidence_interval_95'][0]:.4f}, {stat['confidence_interval_95'][1]:.4f})")
print(f"\n=== Power Analysis ===")
print(f"  Observed effect: {pwr['observed_effect']:.4f}")
print(f"  Achieved power : {pwr['achieved_power']:.1%}")
print(f"  Adequately pow.: {pwr['adequately_powered']}")
print(f"  Required n (80%): {pwr['required_n_for_80pct']}")
print(f"\n=== Sample Size for Δ=0.05, σ=0.15, power=80% ===")
print(f"  Required n per group: {req_n}")